# Debug `RxnDataset` on a periodic NaCl crystal pair

This notebook validates the current crystal-style dataset contract on a
small periodic NaCl example:
- joint reactant+product graph
- sparse intra-fragment radius edges
- periodic `cell` / `pbc` metadata
- `fragment` / `mask` semantics
- node feature layout `h = [pos, one_hot, charge]`
- `charge` interpreted as outer-shell electron count


In [1]:
from pathlib import Path
import json
import sys

import torch
from torch.utils.data import DataLoader

repo_root = Path.cwd()
if not (repo_root / 'dataset').exists() and (repo_root.parent / 'dataset').exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root.parent))
data_dir = repo_root / 'tests' / 'data'
react_file = data_dir / 'nacl_crystal_react.extxyz'
product_file = data_dir / 'nacl_crystal_product.extxyz'

from akmcgc.dataset import RxnDataset

print(f'repo_root = {repo_root}')
print(f'react_file = {react_file}')
print(f'product_file = {product_file}')
assert react_file.exists()
assert product_file.exists()


repo_root = /Users/wx/Desktop/yyxwjq/akmcgc
react_file = /Users/wx/Desktop/yyxwjq/akmcgc/tests/data/nacl_crystal_react.extxyz
product_file = /Users/wx/Desktop/yyxwjq/akmcgc/tests/data/nacl_crystal_product.extxyz


## Debug Contract Helpers

这些 helper 用来打印 dataset 每个字段的 shape、dtype 和 device。


In [2]:
def tensor_contract(name, value):
    if torch.is_tensor(value):
        return f"{name:16s} shape={tuple(value.shape)!s:18s} dtype={str(value.dtype):14s} device={value.device}"
    return f"{name:16s} value={value!r}"


def print_tensor_contract(title, tensors):
    print(f"\n[{title}]")
    for name, value in tensors.items():
        print(tensor_contract(name, value))


In [3]:
from ase.io import read
react_atoms = read(str(react_file))

# Optional visualization. Keep disabled for reproducible non-GUI debug runs.

from ase.visualize import view
view(react_atoms, viewer="x3d")


In [4]:
product_atoms = read(str(product_file))


from ase.visualize import view
view(product_atoms, viewer="x3d")


In [5]:
dataset = RxnDataset(
    react_file=str(react_file),
    product_file=str(product_file),
    cutoff=4.5,
    max_neigh=200,
    r_fixed=True,
    r_pbc=True,
    device='cpu',
)
with torch.no_grad():
    torch.set_printoptions(threshold=float('inf'))
    
sample = dataset[0]
print(sample['edge_index'])

tensor([[ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
          2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
          3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,
          4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,
          5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,  5,
          6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  6,
          7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,  7,
          8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,  8,
          9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,
         10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10,
         11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11, 11,
         12, 12, 12, 12, 12,

In [6]:
print('Loaded one NaCl periodic reaction sample.')
print('Field overview:')
for key, value in sample.items():
    if torch.is_tensor(value):
        print(f"  {key:12s} shape={tuple(value.shape)!s:18s} dtype={str(value.dtype):14s} device={value.device}")
    else:
        print(f"  {key:12s} {value}")


Loaded one NaCl periodic reaction sample.
Field overview:
  h            shape=(32, 122)          dtype=torch.float64  device=cpu
  pos          shape=(32, 3)            dtype=torch.float64  device=cpu
  edge_index   shape=(2, 576)           dtype=torch.int64    device=cpu
  cell_offsets shape=(576, 3)           dtype=torch.float64  device=cpu
  neighbors    shape=(1,)               dtype=torch.int64    device=cpu
  fragment     shape=(32,)              dtype=torch.int64    device=cpu
  mask         shape=(32,)              dtype=torch.int64    device=cpu
  cell         shape=(2, 3, 3)          dtype=torch.float64  device=cpu
  pbc          shape=(2, 3)             dtype=torch.bool     device=cpu
  n_is         16
  n_fs         16


## Sample Tensor Contract

单个样本从 `RxnDataset.__getitem__` 出来后，每个字段的维度和 dtype 如下。


In [7]:
print_tensor_contract('RxnDataset.__getitem__ sample', sample)



[RxnDataset.__getitem__ sample]
h                shape=(32, 122)          dtype=torch.float64  device=cpu
pos              shape=(32, 3)            dtype=torch.float64  device=cpu
edge_index       shape=(2, 576)           dtype=torch.int64    device=cpu
cell_offsets     shape=(576, 3)           dtype=torch.float64  device=cpu
neighbors        shape=(1,)               dtype=torch.int64    device=cpu
fragment         shape=(32,)              dtype=torch.int64    device=cpu
mask             shape=(32,)              dtype=torch.int64    device=cpu
cell             shape=(2, 3, 3)          dtype=torch.float64  device=cpu
pbc              shape=(2, 3)             dtype=torch.bool     device=cpu
n_is             value=16
n_fs             value=16


In [8]:
# Validate collate_fn behavior on two copies of the same sample.
batch = RxnDataset.collate_fn([sample, sample])
n_total = sample['n_is'] + sample['n_fs']

assert batch['h'].shape == (2 * n_total, 122)
assert batch['pos'].shape == (2 * n_total, 3)
assert batch['fragment'].shape == (2 * n_total,)
assert batch['mask'].shape == (2 * n_total,)
assert batch['cell_offsets'].shape[1] == 3
assert batch['neighbors'].shape == (2,)
assert batch['cell'].shape == (2, 2, 3, 3)
assert batch['pbc'].shape == (2, 2, 3)
assert torch.equal(batch['mask'], torch.tensor([0] * n_total + [1] * n_total))

src_b, dst_b = batch['edge_index']
assert torch.all(batch['mask'][src_b] == batch['mask'][dst_b])
assert torch.all(batch['fragment'][src_b] == batch['fragment'][dst_b])

print('Collated two NaCl samples into one disconnected periodic joint graph batch.')
print('Batch field overview:')
for key, value in batch.items():
    if torch.is_tensor(value):
        print(f"  {key:12s} shape={tuple(value.shape)!s:18s} dtype={str(value.dtype):14s} device={value.device}")
    else:
        print(f"  {key:12s} {value}")
print('Batch checks passed.')


Collated two NaCl samples into one disconnected periodic joint graph batch.
Batch field overview:
  h            shape=(64, 122)          dtype=torch.float64  device=cpu
  pos          shape=(64, 3)            dtype=torch.float64  device=cpu
  edge_index   shape=(2, 1152)          dtype=torch.int64    device=cpu
  cell_offsets shape=(1152, 3)          dtype=torch.float64  device=cpu
  neighbors    shape=(2,)               dtype=torch.int64    device=cpu
  fragment     shape=(64,)              dtype=torch.int64    device=cpu
  mask         shape=(64,)              dtype=torch.int64    device=cpu
  cell         shape=(2, 2, 3, 3)       dtype=torch.float64  device=cpu
  pbc          shape=(2, 2, 3)          dtype=torch.bool     device=cpu
  n_is         shape=(2,)               dtype=torch.int64    device=cpu
  n_fs         shape=(2,)               dtype=torch.int64    device=cpu
Batch checks passed.


## Batched Tensor Contract

`RxnDataset.collate_fn` 会把多个样本拼成一个 disconnected joint graph batch。这里检查 batch 后每个字段的维度和 dtype。


In [9]:
print_tensor_contract('RxnDataset.collate_fn batch', batch)



[RxnDataset.collate_fn batch]
h                shape=(64, 122)          dtype=torch.float64  device=cpu
pos              shape=(64, 3)            dtype=torch.float64  device=cpu
edge_index       shape=(2, 1152)          dtype=torch.int64    device=cpu
cell_offsets     shape=(1152, 3)          dtype=torch.float64  device=cpu
neighbors        shape=(2,)               dtype=torch.int64    device=cpu
fragment         shape=(64,)              dtype=torch.int64    device=cpu
mask             shape=(64,)              dtype=torch.int64    device=cpu
cell             shape=(2, 2, 3, 3)       dtype=torch.float64  device=cpu
pbc              shape=(2, 2, 3)          dtype=torch.bool     device=cpu
n_is             shape=(2,)               dtype=torch.int64    device=cpu
n_fs             shape=(2,)               dtype=torch.int64    device=cpu


## Notes

- This example uses a small periodic `NaCl` crystal pair written to `tests/data/`.
- `edge_index` can contain both `i -> j` and `j -> i`, and repeated `(i, j)` pairs when different periodic images of `j` are inside cutoff.
- those repeated pairs are disambiguated by `cell_offsets` and `edge_vec`.
- `charge` here means outer-shell electron count, not partial/formal charge.
